# PyTorch 调试与可复现性

## 学习目标

通过故障复现掌握 shape、dtype、device、梯度、train/eval 和随机种子问题的定位流程。

## 概念模型

先确认输入/输出契约，再确认参数注册和梯度路径，最后检查优化器、学习率和数据。每个实验都包含错误、诊断和修复。

In [ ]:
import torch
from torch import nn
from common.runtime import seed_everything, choose_device

seed_everything(42)
device = choose_device('cpu')
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2)).to(device)
x = torch.randn(8, 4, device=device)
y = torch.randint(0, 2, (8,), device=device)
print(torch.__version__, device, x.shape, x.dtype, y.dtype)

### 实验 1：dtype、shape 和参数注册

`CrossEntropyLoss` 需要类别索引标签；普通 Python list 中的层不会自动注册参数。

In [ ]:
loss_fn = nn.CrossEntropyLoss()
try:
    loss_fn(model(x), y.float())
except RuntimeError as error:
    print('expected dtype error:', type(error).__name__)

bad_layers = nn.Module()
bad_layers.layers = [nn.Linear(4, 4)]
good_layers = nn.ModuleList([nn.Linear(4, 4)])
print('unregistered:', sum(p.numel() for p in bad_layers.parameters()), 'registered:', sum(p.numel() for p in good_layers.parameters()))
assert sum(p.numel() for p in good_layers.parameters()) > 0

### 实验 2：梯度与小 batch 过拟合

训练前先让一个很小的 batch 过拟合，可以快速证明前向、loss、反向和参数更新链路是通的。

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)
losses = []
for _ in range(30):
    optimizer.zero_grad(set_to_none=True)
    loss = loss_fn(model(x), y)
    assert torch.isfinite(loss)
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    losses.append(loss.item())
print('loss:', losses[0], '->', losses[-1], 'grad_norm:', float(grad_norm))
assert losses[-1] < losses[0]

### 实验 3：train/eval、计时和复现

`eval()` 改变 Dropout/BatchNorm 行为，`inference_mode()` 关闭推理计算图。CUDA 计时需要同步；本单元在 CPU 上安全运行。

In [ ]:
drop = nn.Dropout(p=0.5)
drop.train(); train_values = drop(torch.ones(1000))
drop.eval(); eval_values = drop(torch.ones(1000))
assert not torch.equal(train_values, eval_values)
seed_everything(7); first = torch.randn(4)
seed_everything(7); second = torch.randn(4)
torch.testing.assert_close(first, second)
print('train/eval and reproducibility checks passed')

## 检查点

说明为什么 `grad is None`、标签 dtype 错误和 device mismatch 属于不同类别的问题，并写出各自的第一条检查命令。

## 试一试

故意把学习率改成过大值、把标签打乱、把验证阶段改成 `train()`，分别记录 loss、指标或输出的变化。

## 常见错误与调试

固定顺序：打印 shape/dtype/device -> 检查任务与损失匹配 -> 小 batch 过拟合 -> 检查参数和梯度 -> 检查学习率与数据。